# Reproducing Generic Schema Matching with Cupid

- Author: **Ossama Sijbesma (9890238)**
- Date: **18 October 2026**
- Requirements: **[Anaconda 3.14.6](https://www.anaconda.com/download/success?reg=skipped)**

The cupid_tree.py file has partly been generated with Github CoPilot, I have modified the code to match the paper more accurately. Specifically, for the alogrithm to not be greedy (returning only one match for every element) and to include type mapping. This implementation specifically works on hierachical XML files, later in this notebook I also discuss the Valentine library which works primarily on flat structures converted to Pandas.

The paper had an [extended paper](https://www.microsoft.com/en-us/research/wp-content/uploads/2016/02/tr-2001-58.pdf ) in which the experiments have been described in more detail (parameters, datastructures, etc.). Sadly I was unable to find the source code or actual datasets used for the experiments.


In [1]:
# Install dependencies into the active notebook kernel when they are not already available.
# Install packages not included in anaconda
%pip install -q valentine lxml 

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Parse XML and build the tabular inputs used by the comparison matchers.
import xml.etree.ElementTree as ET

import pandas as pd

# Load the tree-based implementation alongside the Valentine baselines.
from cupid_tree import cupid_tree_match, load_schema_tree
from valentine import valentine_match
from valentine.algorithms import Coma, Cupid

### Settings used for this experiment

For the experiment we use synthetic datasets that match the schema from the paper. Those datasets are stored within the dataset folder for which we split the datasets by experiment.

We are using the same parameters as the paper mentioned:

In [3]:
cupid_settings = {
    "th_ns": 0.5,  # Name-similarity threshold, in [0, 1]
    "th_accept": 0.5,  # Acceptance similarity threshold for the final mapping, in [0, 1]
    "th_high": 0.6,  # High-confidence threshold used during structural propagation, in [0, 1]
    "th_low": 0.35,  # Low-confidence threshold used during structural propagation, in [0, 1]
    "c_inc": 1.2,  # Positive reinforcement coefficient for matching children nodes, in [0, +inf)
    "c_dec": 0.9,  # Negative reinforcement coefficient for non-matching children nodes, in [0, +inf)
    "leaf_w_struct": 0.5,  # Weight of structural similarity at leaf level, in [0, 1]
    "w_struct": 0.6,  # Weight of structural similarity at inner-node level, in [0, 1]
}

In [4]:
def run_experiment(experiment_number):

    # Load both schemas from the selected experiment's source and target XML files.
    source_schema = load_schema_tree(f"./datasets/expt{experiment_number}/source.xml")
    target_schema = load_schema_tree(f"./datasets/expt{experiment_number}/target.xml")

    # Display the tree structures so the input to the matcher is visible.
    print("Source Schema:")
    source_schema.show(1)

    print("\nTarget Schema:")
    target_schema.show(1)

    # Perform schema matching using tree-based Cupid
    cupid_matches, _ = cupid_tree_match(source_schema, target_schema, **cupid_settings)

    # Each result contains a source path, target path, and their similarity score.
    print(f"\n{len(cupid_matches)} Cupid matches:")
    for source_path, target_path, score in cupid_matches:
        print(f"  {source_path} <-> {target_path}: {score}")

## Main Experiment

In section 8 of the paper, the authors evaluate the Cupid schema matching algorithm against DIKE and MOMIS (ARTEMIS) across six canonical examples (EXPT 1–6 in this notebook) that try to isolate their matching propertie, and a real-world XML purchase order scenario (EXPT 7: CIDX vs. Excel)

As the paper notes, schema matching is inherently subjective and existing algorithms like DIKE and MOMIS are biased toward schema integration. Therefore, I will reproduce the experiments (EXPT 1–7) using only the Cupid algorithm and compare the results directly against the paper's findings.

### EXPT 01, Indentical schemas

The two schemas, Source and Target, have a single indentical class: Customer.

In [5]:
run_experiment(1)

Source Schema:
  Customer
    Customer_Number : integer
    Name : string
    Address : string

Target Schema:
  Customer
    Customer_Number : integer
    Name : string
    Address : string

3 Cupid matches:
  Customer.Customer_Number <-> Customer.Customer_Number: 1.0
  Customer.Name <-> Customer.Name: 0.986
  Customer.Address <-> Customer.Address: 0.986


Just like in the paper Cupid correctly indentifies the elments without any issue.

### EXPT 02, Atomic elements with same names, but different data types

The telephone attribute is added to both classes: as a string in Source, and as an integer in Target.

In [6]:
run_experiment(2)

Source Schema:
  Customer
    Customer_Number : integer
    Name : string
    Address : string
    Telephone : string

Target Schema:
  Customer
    Customer_Number : integer
    Name : string
    Address : string
    Telephone : integer

5 Cupid matches:
  Customer.Customer_Number <-> Customer.Customer_Number: 1.0
  Customer.Name <-> Customer.Name: 0.986
  Customer.Address <-> Customer.Address: 0.986
  Customer.Telephone <-> Customer.Telephone: 0.646
  Customer.Customer_Number <-> Customer.Telephone: 0.5


We use a type compatibilty table in our algorithm and surely enough we see that Telephone matches, but the score is really low. The low acceptance threshold does cause Customer_Number to match Telephone, showing that we can have a 1:n cardinality. But in case of multiple mappings we select the highest score. 

### EXPT 03, Atomic elements with same data types, but different names (a prefix or suffix is added)

A prefix or suffix is added to each of the names in the target schema.

In [7]:
run_experiment(3)

Source Schema:
  Customer
    Customer_Number : integer
    Name : string
    Address : string

Target Schema:
  Customer
    CustomerNumber : integer
    CustomerName : string
    StreetAddress : string

5 Cupid matches:
  Customer.Customer_Number <-> Customer.CustomerNumber: 1.0
  Customer.Name <-> Customer.CustomerName: 0.875
  Customer.Address <-> Customer.StreetAddress: 0.861
  Customer.Customer_Number <-> Customer.CustomerName: 0.509
  Customer.Address <-> Customer.CustomerName: 0.5


Again all fields seem to match correctly, meaning that Cupid is tolerant of name variations.

### EXPT 04, Different class names, but atomic elements same names and data types

In the target schema we have schanged the class name to Person.

In [8]:
run_experiment(4)

Source Schema:
  Customer
    Customer_Number : integer
    Name : string
    Address : string

Target Schema:
  Person
    Customer_Number : integer
    Name : string
    Address : string

3 Cupid matches:
  Customer.Customer_Number <-> Person.Customer_Number: 1.0
  Customer.Name <-> Person.Name: 0.905
  Customer.Address <-> Person.Address: 0.905


The scores are slighly lower than in the first experiment, but regardless very high. This indicates that leaf level comparisons work really well. In the bonus section at the end of this paper we tell a bit more about this together with the next experiment which introduces nesting.

### EXPT 05, Different Nesting of the data – similar schemas with nested and flat structures

Like mentioned before, we now introduce nesting. Nesting in the schema are indicated with tabs (see name and address) in the output below.

In [9]:
run_experiment(5)

Source Schema:
  Customer
    SSN : integer
    Telephone : string
    Name
      FirstName : string
      LastName : string
    Address
      Street : string
      City : string
      State : string
      Zip : integer

Target Schema:
  Customer
    SSN : integer
    Telephone : string
    FirstName : string
    LastName : string
    Street : string
    City : string
    State : string
    Zip : integer

10 Cupid matches:
  Customer.SSN <-> Customer.SSN: 0.986
  Customer.Telephone <-> Customer.Telephone: 0.986
  Customer.Name.FirstName <-> Customer.FirstName: 0.986
  Customer.Name.LastName <-> Customer.LastName: 0.986
  Customer.Address.Street <-> Customer.Street: 0.937
  Customer.Address.City <-> Customer.City: 0.937
  Customer.Address.State <-> Customer.State: 0.937
  Customer.Address.Zip <-> Customer.Zip: 0.937
  Customer.Name.FirstName <-> Customer.LastName: 0.736
  Customer.Name.LastName <-> Customer.FirstName: 0.736


Cupid is able to match the elements, but notice how the score is lower than with the direct matches (e.g. SSN and Telephone).

### EXPT 06, Type Substitution or Context dependent mapping

We now have two clases which contain exactly the same data (an address) in the Source and in the Target (ShippingAddress, BillingAddress).

In [10]:
run_experiment(6)

Source Schema:
  PurchaseOrder
    OrderNumber : integer
    ProductName : string
    ShippingAddress
      Address
        Name : string
        Street : string
        City : string
        Zip : integer
        Telephone : string
    BillingAddress
      Address
        Name : string
        Street : string
        City : string
        Zip : integer
        Telephone : string

Target Schema:
  PurchaseOrder
    OrderNumber : integer
    ProductName : string
    ShippingAddress
      ShipTo
        Name : string
        Street : string
        City : string
        Zip : integer
        Telephone : string
    BillingAddress
      BillTo
        Name : string
        Street : string
        City : string
        Zip : integer
        Telephone : string

50 Cupid matches:
  PurchaseOrder.ShippingAddress.Address.Name <-> PurchaseOrder.ShippingAddress.ShipTo.Name: 1.0
  PurchaseOrder.ShippingAddress.Address.Street <-> PurchaseOrder.ShippingAddress.ShipTo.Street: 1.0
  PurchaseOrder.Ship

We can really notice here that strucutral similarity in this example:

-   PurchaseOrder.ShippingAddress.Address.Name <-> PurchaseOrder.ShippingAddress.ShipTo.Name: 1.0
-   PurchaseOrder.ShippingAddress.Address.Name <-> PurchaseOrder.BillingAddress.BillTo.Name: 0.959

The difference here is within the inner nodes of the tree, resulting in a better score for the first one than the second one. Taking the highest score of the two you would get the correct mappings.

### EXPT 07, CIDX vs. Excel

In the experiment below we actually get a real world example, where we use two XML purchase orders (CIDX and Excel).
As the dataset is pretty large we are also provided with the mappings that 

In [11]:
run_experiment(7)

Source Schema:
  PO
    POHeader
      PODate : date
      PONumber : integer
    Contact
      ContactName : string
      ContactFunctionCode : string
      ContactEmail : string
      ContactPhone : string
    POShipTo
      entityIdentifier : string
      attn : string
      Street1 : string
      Street2 : string
      Street3 : string
      Street4 : string
      City : string
      StateProvince : string
      PostalCode : integer
      Country : string
    POBillTo
      entityIdentifier : string
      attn : string
      Street1 : string
      Street2 : string
      Street3 : string
      Street4 : string
      City : string
      StateProvince : string
      PostalCode : integer
      Country : string
    POLines
      count : integer
      startAt : integer
      Item
        line : integer
        qty : integer
        uom : string
        unitPrice : float
        partno : string

Target Schema:
  PurchaseOrder
    Header
      orderNum : integer
      orderDate : string
  


50 Cupid matches:
  PO.Contact.ContactName <-> PurchaseOrder.Contact.contactName: 0.865
  PO.POShipTo.Street1 <-> PurchaseOrder.DeliverTo.Address.street1: 0.819
  PO.POShipTo.Street1 <-> PurchaseOrder.InvoiceTo.Address.street1: 0.819
  PO.POShipTo.Street2 <-> PurchaseOrder.DeliverTo.Address.street2: 0.819
  PO.POShipTo.Street2 <-> PurchaseOrder.InvoiceTo.Address.street2: 0.819
  PO.POShipTo.Street3 <-> PurchaseOrder.DeliverTo.Address.street3: 0.819
  PO.POShipTo.Street3 <-> PurchaseOrder.InvoiceTo.Address.street3: 0.819
  PO.POShipTo.Street4 <-> PurchaseOrder.DeliverTo.Address.street4: 0.819
  PO.POShipTo.Street4 <-> PurchaseOrder.InvoiceTo.Address.street4: 0.819
  PO.POShipTo.City <-> PurchaseOrder.DeliverTo.Address.city: 0.819
  PO.POShipTo.City <-> PurchaseOrder.InvoiceTo.Address.city: 0.819
  PO.POShipTo.StateProvince <-> PurchaseOrder.DeliverTo.Address.stateProvince: 0.819
  PO.POShipTo.StateProvince <-> PurchaseOrder.InvoiceTo.Address.stateProvince: 0.819
  PO.POShipTo.PostalCod

So although here we see that we have a small difference with the results in the paper, and it is explainable as well.

For Cupid, the thesauri had a total of 4 abbreviations (UOM, PO, Qty, Num) and 2 synonym entries (Invoice,Bill; Ship,Deliver) that were added for the example. I therefore did some preprocessing on the data to create a dataset where those abbreviations and synonyms match the values. See the run below for results:

In [12]:
run_experiment("7b")

Source Schema:
  PurchaseOrder
    Header
      Date : date
      Number : integer
    Contact
      ContactName : string
      ContactFunctionCode : string
      ContactEmail : string
      ContactPhone : string
    DeliverTo
      entityIdentifier : string
      attn : string
      Street1 : string
      Street2 : string
      Street3 : string
      Street4 : string
      City : string
      StateProvince : string
      stalCode : integer
      Country : string
    BillTo
      entityIdentifier : string
      attn : string
      Street1 : string
      Street2 : string
      Street3 : string
      Street4 : string
      City : string
      StateProvince : string
      stalCode : integer
      Country : string
    Lines
      count : integer
      startAt : integer
      Item
        line : integer
        Quantity : integer
        unitOfMeasure : string
        unitPrice : float
        partno : string

Target Schema:
  PurchaseOrder
    Header
      orderNumber : integer
      order

Manual preprocessing increased the total match count from 50 to 61, displayinh the impact of the synonym and abbreviation translation dictionary on performance.

The authors also take this moments to note the presence of false positives, which are the cases with 1:n cardinality relationships (e.g., PurchaseOrder.Contact.ContactName). This behavior is expected given the design of the algorithm. Interestingly, Copilot originally implemented a greedy approach that enforced 1:1 cardinality. However, this was changed to follow the paper's original implementation more closely.

### Conclusion

Although the paper wasn't provided with the exact source code, I was able to reproduce the experiments using the details provided by the authors to an extend that the possibilities are validated. I really like their approach of viewing schemas as trees and matching the nodes both the linguistic and structural levels. Most uncertainty in my code likely comes from linguistic matching, as normalization, categorization, and comparison can be achieved in multiple ways. Additionally, the synonym and abbreviation dictionary (thesaurus) heavily influences the performance. Therefore, using a domain-specific dictionary could be beneficial if you work in an environment with specific environment specific abbreviations.

One of the things I did find odd was the parameters the authors selected, the threshold to get a match was fairly low (0.5). This made me realize that it was quick to match two schemas in case of little similarity, thus causing a the possibility for higher false positives. As this experiment was mainly a showcase I do believe that it wasn't a big issue but in case of real world situations I would tune this parameter to the higher side (unless you would benefit from that).

## Bonus

While selecting an assignment paper, I came across the [Valentine framework](https://github.com/delftdata/valentine/tree/master/valentine/algorithms/cupid), which includes Cupid as one of its schema matchers. Therefore I tried to performing the main experiment on this library. However, I soon noticed that it would fail on nested structures. That is because the library creates a tree structure from a flat structure (pandas dataframe). As a workaround I tried to convert the XML files into a flat structure. This kinda works but you will be penalized as it benefits less from structural similarty, that is per my understanding. Anyway see an example below:

In [ ]:
def xml_to_wide_df(xml_path, table_name):
    """Flatten an XML schema into a wide, empty DataFrame (one column per leaf path)."""
    # Convert each terminal XML element into a fully qualified column name.
    root = ET.parse(xml_path).getroot()
    leaf_paths = []

    def walk(node, path):
        # Internal elements extend the path; elements with no children become columns.
        children = list(node)
        if not children:
            leaf_paths.append(path)
            return
        for child in children:
            walk(child, f"{path}.{child.get('name', child.tag)}")

    walk(root, root.get("name", root.tag))
    # Valentine uses the column names as schema attributes; no data rows are needed here.
    df = pd.DataFrame(columns=leaf_paths)
    df.name = table_name
    return df


def run_valentine_experiment(experiment_number):
    # Adapt the XML schemas to the tabular input expected by Valentine.
    src_df = xml_to_wide_df(f"./datasets/expt{experiment_number}/source.xml", "source")
    tgt_df = xml_to_wide_df(f"./datasets/expt{experiment_number}/target.xml", "target")

    print("Source columns:", list(src_df.columns))
    print("Target columns:", list(tgt_df.columns))

    # Run Valentine's Cupid implementation against the two schema-only tables.
    matcher = Cupid(**cupid_settings)
    matches = valentine_match([src_df, tgt_df], matcher)

    print("\nValentine Cupid Matches:")
    for pair, sim in matches.items():
        print(f"  {pair.source_column} <-> {pair.target_column}: {round(sim, 4)}")
    return matches

In [14]:
expt_valentine_matches = run_valentine_experiment(1)

Source columns: ['Schema1.Customer.Customer_Number', 'Schema1.Customer.Name', 'Schema1.Customer.Address', 'Schema1.Customer.Customer_Number', 'Schema1.Customer.Name', 'Schema1.Customer.Address', 'Schema1.Customer.Customer_Number', 'Schema1.Customer.Name', 'Schema1.Customer.Address']
Target columns: ['Schema2.Customer.Customer_Number', 'Schema2.Customer.Name', 'Schema2.Customer.Address', 'Schema2.Customer.Customer_Number', 'Schema2.Customer.Name', 'Schema2.Customer.Address', 'Schema2.Customer.Customer_Number', 'Schema2.Customer.Name', 'Schema2.Customer.Address']

Valentine Cupid Matches:
  Schema1.Customer.Customer_Number <-> Schema2.Customer.Customer_Number: 0.7321
  Schema1.Customer.Name <-> Schema2.Customer.Name: 0.7262
  Schema1.Customer.Address <-> Schema2.Customer.Address: 0.7262
  Schema1.Customer.Customer_Number <-> Schema2.Customer.Name: 0.7092
  Schema1.Customer.Name <-> Schema2.Customer.Customer_Number: 0.7092
  Schema1.Customer.Name <-> Schema2.Customer.Address: 0.6706
  Sch

In [15]:
expt_valentine_matches = run_valentine_experiment(5)

Source columns: ['Nested-Schema.Customer.SSN', 'Nested-Schema.Customer.Telephone', 'Nested-Schema.Customer.Name.FirstName', 'Nested-Schema.Customer.Name.LastName', 'Nested-Schema.Customer.Address.Street', 'Nested-Schema.Customer.Address.City', 'Nested-Schema.Customer.Address.State', 'Nested-Schema.Customer.Address.Zip', 'Nested-Schema.Customer.SSN', 'Nested-Schema.Customer.Telephone', 'Nested-Schema.Customer.Name.FirstName', 'Nested-Schema.Customer.Name.LastName', 'Nested-Schema.Customer.Address.Street', 'Nested-Schema.Customer.Address.City', 'Nested-Schema.Customer.Address.State', 'Nested-Schema.Customer.Address.Zip', 'Nested-Schema.Customer.SSN', 'Nested-Schema.Customer.Telephone', 'Nested-Schema.Customer.Name.FirstName', 'Nested-Schema.Customer.Name.LastName', 'Nested-Schema.Customer.Address.Street', 'Nested-Schema.Customer.Address.City', 'Nested-Schema.Customer.Address.State', 'Nested-Schema.Customer.Address.Zip']
Target columns: ['Flat-Schema.Customer.SSN', 'Flat-Schema.Customer.T

### Coma

Valentine also supports usage of the COMA (COmbination of MAtching algorithms) 3.0 schema matching algorithm, which I understand is the strongest default. Just like Cupid it uses a combination of sub matchers. So it would be intressting to see how it performs as well.

In [ ]:
# Read the experiment's XML schemas as tables for Valentine's COMA matcher.
source_schema = pd.read_xml("./datasets/expt1/source.xml")
target_schema = pd.read_xml("./datasets/expt1/target.xml")

# Perform schema matching with COMA
coma_matches = valentine_match([source_schema, target_schema], Coma())

print("\nValentine COMA Matches:")
for pair, sim in coma_matches.items():
    print(f"  {pair.source_column} <-> {pair.target_column}: {round(sim, 4)}")


Valentine COMA Matches:
  Customer_Number <-> Customer_Number: 1.0
  Name <-> Name: 1.0
  Address <-> Address: 1.0
